In [1]:
from tensorflow.keras.applications import VGG16
import tensorflow as tf
import numpy as np
import os
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models

2025-05-06 07:41:17.690195: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-06 07:41:17.700925: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-06 07:41:17.930139: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-06 07:41:22.998663: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-poin

In [2]:
def preprocess_img(img_path, size=224):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((size, size))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.keras.applications.vgg16.preprocess_input(img_array)
    return np.expand_dims(img_array, axis=0)

In [3]:
class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(mean))
        return mean + tf.exp(log_var*0.5) * epsilon

def create_vae_encoder(latent_dim=64):
    encoder_inputs = tf.keras.Input(shape=(64, 64, 3))
    x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
    x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    mean = layers.Dense(latent_dim)(x)
    log_var = layers.Dense(latent_dim)(x)
    latent_vect = Sampling()([mean, log_var])
    encoder = tf.keras.Model(encoder_inputs, latent_vect, name="encoder")
    return encoder

vae_encoder = create_vae_encoder()

2025-05-06 07:41:25.758068: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
def extract_features(img_path, model, size=224):
    img_tensor = preprocess_img(img_path, size)
    features = model.predict(img_tensor)
    return features.flatten()

In [5]:
def load_feature_dataset(root_dir, model, size=224):
    X = []  
    y = [] 
    trial_metadata = []

    for rule_type in os.listdir(root_dir):
        rule_path = os.path.join(root_dir, rule_type)
        if not os.path.isdir(rule_path): continue

        for rule_folder in os.listdir(rule_path):
            img_folder = os.path.join(rule_path, rule_folder)
            if not os.path.isdir(img_folder): continue

            img_paths = [
                os.path.join(img_folder, "inlier_0.png"),
                os.path.join(img_folder, "inlier_1.png"),
                os.path.join(img_folder, "inlier_2.png"),
                os.path.join(img_folder, "outlier.png")
            ]

            features = [extract_features(p, model, size) for p in img_paths]
            paired = list(zip(img_paths, features))
            np.random.shuffle(paired)

            shuffled_paths, shuffled_features = zip(*paired)
            shuffled_features = list(shuffled_features)

            outlier_idx = [i for i, path in enumerate(shuffled_paths) if "outlier" in path][0]

            X.append(shuffled_features)
            y.append(outlier_idx)
            
            trial_metadata.append({
                "rule": rule_type,
                "img_paths": shuffled_paths,
                "true_outlier_idx": outlier_idx,
            })

    return np.array(X), np.array(y), trial_metadata


In [6]:
model = vae_encoder
X, y, trial_metadata = load_feature_dataset("data", model, size=64)
X_flat = X.reshape((X.shape[0], -1))
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
X_train, X_test = X_flat[train_idx], X_flat[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━

In [8]:
clf = MLPClassifier(hidden_layer_sizes=(128,), activation='relu', max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Odd-one-out accuracy (MLP):", acc)

Odd-one-out accuracy (MLP): 0.2
